# 05 — Colab RL Training: Identity Compressor Baseline

**Columbia MSDS STATGR5293 — Optimized LLM Planning with Memory**

This notebook runs the full PPO training pipeline on Google Colab using the **identity compressor** baseline. It covers:

1. Environment setup (repo clone, pip install, API keys)
2. Generating synthetic `UserRequest` files from the travel simulator
3. Smoke-testing a single episode before training
4. Launching PPO training with TensorBoard monitoring
5. Live diagnostics: reward curves, constraint satisfaction, reward predictor weights
6. Saving/loading checkpoints to Google Drive
7. Running deterministic evaluation on the trained policy

**Runtime:** Set to `GPU → T4` (Runtime → Change runtime type). Training ~50 k steps takes ≈ 20–40 min depending on episode length.

---
## 1  Environment Setup

In [ ]:
# Verify GPU availability
import subprocess, sys
result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                        capture_output=True, text=True)
if result.returncode == 0:
    print('GPU:', result.stdout.strip())
else:
    print('WARNING: No GPU detected. Training will be slow on CPU.')
    print('Go to Runtime → Change runtime type → GPU → T4')

In [ ]:
import os

REPO_URL = 'https://github.com/YOUR_ORG/optimized-llm-planning-memory.git'  # update before use
REPO_DIR = '/content/optimized-llm-planning-memory'

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    print('Repo already cloned. Pulling latest...')
    !git -C {REPO_DIR} pull

%cd {REPO_DIR}
print('Working directory:', os.getcwd())

In [ ]:
# Install project + travel_world simulator
# travel_world must be pip-installable from wherever the team hosts it.
# Replace the placeholder URL with the actual wheel / git URL.
!pip install -q -e '.[dev]'
!pip install -q travel_world   # or: pip install git+https://github.com/YOUR_ORG/travel_world.git
!pip install -q tensorboard
print('Installation complete.')

---
## 2  API Keys

Store secrets with Colab's built-in **Secrets** panel (🔑 icon in the left sidebar) rather than pasting them into cells.

Required secrets:
| Secret name | Value |
|---|---|
| `ANTHROPIC_API_KEY` | `sk-ant-...` |
| `OPENAI_API_KEY` | `sk-...` (optional, for OpenAI models) |

The cell below reads them into environment variables.

In [ ]:
from google.colab import userdata

def _load_secret(name: str) -> str | None:
    try:
        val = userdata.get(name)
        os.environ[name] = val
        print(f'  {name}: loaded ({len(val)} chars)')
        return val
    except Exception:
        print(f'  {name}: NOT FOUND — add it via the 🔑 Secrets panel')
        return None

print('Loading API keys from Colab Secrets...')
_load_secret('ANTHROPIC_API_KEY')
_load_secret('OPENAI_API_KEY')  # optional

# Which LLM the planning agent will call
os.environ.setdefault('AGENT_LLM_MODEL_ID', 'anthropic/claude-haiku-4-5-20251001')
print('Agent LLM:', os.environ['AGENT_LLM_MODEL_ID'])

---
## 3  Google Drive Mount (for checkpoint persistence)

Colab sessions reset after ~12 h of inactivity. Mount Drive so checkpoints survive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

# All outputs will be written here AND mirrored to Drive
DRIVE_OUTPUT_DIR = '/content/drive/MyDrive/optllm_training'
os.makedirs(DRIVE_OUTPUT_DIR, exist_ok=True)

# Symlink so Hydra outputs go straight to Drive
LOCAL_OUTPUT = '/content/optimized-llm-planning-memory/outputs'
if not os.path.exists(LOCAL_OUTPUT):
    os.symlink(DRIVE_OUTPUT_DIR, LOCAL_OUTPUT)
    print(f'Symlinked outputs/ → {DRIVE_OUTPUT_DIR}')
else:
    print('outputs/ already exists (previous run or local path)')

---
## 4  Configuration

All knobs live in `configs/`. The Colab run uses:
- `compressor=identity` — no distillation; PPO trains a scalar dummy parameter
- `training=ppo_colab` — 2 envs, 50 k steps, small batch (fits T4 VRAM)

Override any value by adding `key=value` to the training cell's `!python` call.

In [ ]:
# Preview the active config without running training
!python scripts/run_training.py \
    compressor=identity \
    training=ppo_colab \
    --cfg job   # prints resolved config and exits

---
## 5  Generate User Requests

Creates `data/user_requests/{train,val,test}/*.json` using real city IDs from the travel simulator. No LLM call is needed.

In [ ]:
!python scripts/generate_user_requests.py \
    n_train=40 n_val=10 n_test=10 \
    project.seed=42

# Verify
from pathlib import Path
for split in ('train', 'val', 'test'):
    n = len(list(Path(f'data/user_requests/{split}').glob('*.json')))
    print(f'  {split}: {n} requests')

---
## 6  Smoke Test — Single Episode

Run one episode end-to-end before committing to a full training run. A successful run prints the episode summary and saves a JSON log to `outputs/episodes/`.

In [ ]:
!python scripts/run_episode.py \
    compressor=identity \
    agent.max_steps=5 \
    project.seed=42

In [ ]:
import json
from pathlib import Path

episode_files = sorted(Path('outputs/episodes').glob('*.json'))
if not episode_files:
    print('No episode logs found — check the error output above.')
else:
    log = json.loads(episode_files[-1].read_text())
    print(f"episode_id : {log['episode_id']}")
    print(f"success    : {log['success']}")
    print(f"total_steps: {log['total_steps']}")
    print(f"error      : {log.get('error')}")
    rc = log.get('reward_components', {})
    print(f"total_reward          : {rc.get('total_reward', 0):.3f}")
    print(f"hard_constraint_score : {rc.get('hard_constraint_score', 0):.3f}")
    print(f"tool_efficiency_score : {rc.get('tool_efficiency_score', 0):.3f}")

---
## 7  TensorBoard

Launch TensorBoard **before** training so it updates live. Metrics logged per episode:

| Tag | Meaning |
|---|---|
| `episode/total_reward` | Composite reward (primary signal) |
| `episode/hard_constraint_score` | Fraction of hard constraints satisfied |
| `episode/soft_constraint_score` | Weighted soft-constraint score |
| `episode/tool_efficiency_score` | Fraction of tool calls that succeeded |
| `episode/tool_failure_penalty` | Penalty for failed/repeated tool calls |
| `train/policy_gradient_loss` | PPO policy loss (SB3 built-in) |
| `train/value_loss` | PPO value function loss |
| `train/entropy_loss` | Policy entropy (exploration) |

In [ ]:
%load_ext tensorboard
%tensorboard --logdir outputs/logs

---
## 8  PPO Training

This cell is the main training run. It will:
- Create `training.n_envs` parallel `CompressionEnv` instances
- Run PPO for `training.num_timesteps` steps
- Save SB3 checkpoints to `outputs/checkpoints/` every 5 k steps
- Save compressor weights alongside each checkpoint
- Feed episode data to the `RewardPredictorCallback` every 50 episodes

Interrupt with **Runtime → Interrupt execution**; the last checkpoint is preserved.

In [ ]:
!python scripts/run_training.py \
    compressor=identity \
    training=ppo_colab \
    project.seed=42 \
    project.run_name=identity_baseline_$(date +%Y%m%d_%H%M)

### Resuming from a checkpoint

If the session restarted, re-run cells 1–5, then run the cell below to resume PPO training from the last saved checkpoint.

In [ ]:
from pathlib import Path

ckpt_dir = Path('outputs/checkpoints')
zips = sorted(ckpt_dir.glob('ppo_compressor_*_steps.zip'))
if not zips:
    print('No checkpoints found — run training first.')
else:
    latest = zips[-1]
    print(f'Resuming from: {latest}')
    !python scripts/run_training.py \
        compressor=identity \
        training=ppo_colab \
        training.resume_from={latest} \
        project.seed=42

---
## 9  Diagnostics

Parse episode logs written during training and plot the key metrics.

In [ ]:
import json
from pathlib import Path
import pandas as pd

episode_dir = Path('outputs/episodes')
records = []
for f in sorted(episode_dir.glob('*.json')):
    try:
        log = json.loads(f.read_text())
        rc = log.get('reward_components', {})
        records.append({
            'episode_id'             : log['episode_id'],
            'success'                : log['success'],
            'total_steps'            : log['total_steps'],
            'total_reward'           : rc.get('total_reward', 0),
            'hard_constraint_score'  : rc.get('hard_constraint_score', 0),
            'soft_constraint_score'  : rc.get('soft_constraint_score', 0),
            'tool_efficiency_score'  : rc.get('tool_efficiency_score', 0),
            'tool_failure_penalty'   : rc.get('tool_failure_penalty', 0),
        })
    except Exception as e:
        print(f'Skipping {f.name}: {e}')

df = pd.DataFrame(records)
print(f'Loaded {len(df)} episode logs')
df.tail(10)

In [ ]:
import matplotlib.pyplot as plt

if df.empty:
    print('No data to plot.')
else:
    fig, axes = plt.subplots(2, 2, figsize=(14, 8))
    fig.suptitle('RL Training Diagnostics — Identity Compressor', fontsize=14)

    window = max(1, len(df) // 20)  # rolling window ≈ 5% of episodes

    for ax, col, title in [
        (axes[0, 0], 'total_reward',           'Total Reward'),
        (axes[0, 1], 'hard_constraint_score',  'Hard Constraint Score'),
        (axes[1, 0], 'soft_constraint_score',  'Soft Constraint Score'),
        (axes[1, 1], 'tool_efficiency_score',  'Tool Efficiency Score'),
    ]:
        ax.plot(df[col].values, alpha=0.3, color='steelblue', linewidth=0.8)
        ax.plot(df[col].rolling(window, min_periods=1).mean().values,
                color='steelblue', linewidth=2, label=f'MA-{window}')
        ax.set_title(title)
        ax.set_xlabel('Episode')
        ax.legend()
        ax.grid(alpha=0.3)

    plt.tight_layout()
    plt.savefig('outputs/training_diagnostics.png', dpi=150)
    plt.show()
    print('Saved → outputs/training_diagnostics.png')

In [ ]:
# Success rate and constraint satisfaction summary
if not df.empty:
    last_n = min(50, len(df))
    recent = df.tail(last_n)
    print(f'=== Last {last_n} episodes ===')
    print(f'  Success rate            : {recent["success"].mean():.1%}')
    print(f'  Avg total reward        : {recent["total_reward"].mean():.3f}')
    print(f'  Avg hard constraint     : {recent["hard_constraint_score"].mean():.3f}')
    print(f'  Avg soft constraint     : {recent["soft_constraint_score"].mean():.3f}')
    print(f'  Avg tool efficiency     : {recent["tool_efficiency_score"].mean():.3f}')
    print(f'  Avg steps per episode   : {recent["total_steps"].mean():.1f}')

### Reward predictor diagnostics

The `RewardPredictorComponent` is a small `nn.Linear(5, 1)` model trained with Adam/MSE every 50 episodes to predict episode reward from 5 features. Inspect the learned weights to sanity-check which features the linear model finds most predictive.

In [ ]:
import torch
from pathlib import Path

# The RewardPredictorCallback saves weights alongside the compressor checkpoint
rp_files = sorted(Path('outputs/checkpoints').rglob('reward_predictor.pt'))
if not rp_files:
    print('No reward predictor checkpoint yet (fit() requires ≥ 20 episodes).')
else:
    ckpt = torch.load(rp_files[-1], map_location='cpu')
    weight = ckpt['model_state']['weight'].squeeze()   # shape (5,)
    bias   = ckpt['model_state']['bias'].item()
    feature_names = ckpt['feature_names']
    print('RewardPredictorComponent weights (latest checkpoint):')
    for name, w in zip(feature_names, weight.tolist()):
        print(f'  {name:<30s}  {w:+.4f}')
    print(f'  bias                            {bias:+.4f}')
    print(f'  trained on {ckpt["n_episodes_trained"]} episodes')

---
## 10  Checkpoint Management

In [ ]:
from pathlib import Path

ckpt_dir = Path('outputs/checkpoints')
zips = sorted(ckpt_dir.glob('ppo_compressor_*_steps.zip'))
print(f'SB3 checkpoints ({len(zips)} total):')
for z in zips:
    size_mb = z.stat().st_size / 1e6
    print(f'  {z.name:<50s}  {size_mb:.1f} MB')

final = ckpt_dir / 'final'
if final.exists():
    print(f'\nFinal checkpoint: {final}')
    for f in sorted(final.rglob('*')):
        if f.is_file():
            print(f'  {f.relative_to(final)}')

In [ ]:
# Verify that a compressor checkpoint loads correctly
import sys
sys.path.insert(0, 'src')

from pathlib import Path
from optimized_llm_planning_memory.compressor.identity_compressor import IdentityCompressor

ckpt_path = Path('outputs/checkpoints/final/compressor')
if ckpt_path.exists():
    ic = IdentityCompressor()
    ic.load_checkpoint(str(ckpt_path))
    print('Checkpoint loaded successfully.')
    print(f'  _dummy_param value: {ic._dummy_param.item():.6f}')
else:
    print(f'Checkpoint not found at {ckpt_path}. Run training first.')

---
## 11  Evaluation

Run deterministic evaluation (no LLM judge required) on the test set. Results are written to `outputs/eval_results/`.

In [ ]:
!python scripts/run_evaluation.py \
    compressor=identity \
    eval.deterministic_only=true \
    training.resume_from=outputs/checkpoints/final/ppo_model.zip

In [ ]:
from pathlib import Path
import json

eval_files = sorted(Path('outputs/eval_results').glob('*.json'))
if not eval_files:
    print('No eval results yet.')
else:
    results = json.loads(eval_files[-1].read_text())
    print(json.dumps(results, indent=2))

---
## Appendix — Troubleshooting

| Problem | Fix |
|---|---|
| `ModuleNotFoundError: travel_world` | Re-run the pip install cell; confirm the package URL is correct |
| `ANTHROPIC_API_KEY not found` | Add it via the 🔑 Secrets panel; re-run the secrets cell |
| TensorBoard shows no data | Training must write at least one episode; check the training cell output for errors |
| CUDA out of memory | Reduce `training.ppo.batch_size` to 16 or set `training.n_envs=1` |
| Session disconnected mid-run | Outputs are mirrored to Drive; re-run from the **Resuming from a checkpoint** cell |
| Episode always fails with max_steps | Increase `agent.max_steps` or check that `AGENT_LLM_MODEL_ID` has a valid API key |

---
## 12  Bundle & Share This Run

Package the training run into a single  for download or sharing.
This bundles: , JSONL logs, and the final checkpoint.

The  records **exactly** which config produced this run, so
notebooks/08_run_comparison.ipynb can compare runs without guessing.


In [ ]:
import sys
sys.path.insert(0, "src")

from pathlib import Path
from optimized_llm_planning_memory.training.run_manifest import list_manifests
from optimized_llm_planning_memory.utils.colab_utils import (
    bundle_run, upload_to_drive, download_bundle, estimate_run_size
)

# Find the latest training run
manifests = list_manifests("outputs/training")
if not manifests:
    print("No training runs found. Complete training first.")
else:
    latest = manifests[0]  # newest first
    run_id = latest.run_id
    print(f"Latest run : {run_id}")
    print(f"Compressor : {latest.compressor_type}")
    print(f"Run name   : {latest.run_name}")
    print(f"Timesteps  : {latest.num_timesteps:,}")
    print()

    # Estimate storage before bundling
    sizes = estimate_run_size(run_id)
    for k, v in sizes.items():
        print(f"  {k:<25} {v:.1f} MB")


In [ ]:
# Create the bundle archive
# The archive includes: manifest.json, ppo_metrics.jsonl,
# episode_metrics.jsonl, final checkpoint, and TensorBoard logs (if < 50 MB).

bundle_path = bundle_run(
    run_id=run_id,
    output_dir="outputs",
    bundle_dir="outputs/bundles",
)
print(f"Bundle: {bundle_path}")


In [ ]:
# Upload bundle to Google Drive (optional)
# Other team members can download this and unpack it locally for comparison.

DRIVE_BUNDLES_DIR = "/content/drive/MyDrive/optllm_training/bundles"

upload_to_drive(bundle_path, drive_dir=DRIVE_BUNDLES_DIR)

# Trigger in-browser download (works in Colab)
download_bundle(run_id, bundle_dir="outputs/bundles")


### Running evaluation directly from a run_id

Instead of manually specifying a checkpoint path, use  to let the
evaluation script resolve the checkpoint automatically from :



The script reads  to find the
checkpoint and infer the compressor type — no manual path needed.
